# 🎯 [PRC07_M09] AAR 기반 대체선 오추천 사례 분석 및 부품 유사도 매칭 가중치 재보정 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M09 |
| **미션 주제** | AAR 기반 대체선 오추천 사례 분석 및 부품 유사도 매칭 가중치 재보정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 09. AAR 기반 대체선 오추천 사례 분석 및 부품 유사도 매칭 가중치 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 대체선 오추천 사례를 분석하고 매칭 가중치를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR로 대체선 오추천 사례를 분석하고 매칭 가중치를 재보정하되 별도 검증 세트로 개선 여부를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **현장 모의 AAR 세트에서 1순위 오추천을 원인별(후보 충돌, 크기 편차 감점 등)로 분류하고, 가중치 재보정, 정밀 규격 항목 추가, 후보 1:1 배정, 근소 차이 검토 표시를 개선안으로 시험 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/prc07_uat_feedback_report.md 및 패치 내역` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 실무 업무 생산성 기여도 및 현장 적응성

#### 미션 PASS 조건

- 정량 기준: UAT 체크리스트 만족도 90점 이상, 개선 요구 5건 반영 완료


- 진행 기준: UAT 결과 보고서 작성 및 최종 버그 픽스 완료


- 검증 방법: 실제 업무 시나리오 기반 최종 검수 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""현장 모의 AAR 세트에서 대체 부품 1순위 오추천 사례를 원인별로 분류하고, 가중치 재보정·정밀 규격 항목·후보 1:1 배정·근소 차이 검토 표시 등 개선안을 별도 검증 세트로 비교해 채택 여부를 정한 뒤, UAT 체크리스트 점수와 개선 요구 반영 현황을 계산한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    # [TODO : 학생 실습 9] AAR 기반 대체선 오추천 사례 분석 및 부품 유사도 매칭 가중치 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sig(x):
    # [TODO : 학생 실습 9] AAR 기반 대체선 오추천 사례 분석 및 부품 유사도 매칭 가중치 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 대체 부품 추천 공통 함수: 텍스트 유사도 + 규격 적합성 + 지식 그래프 이웃 유사도 ----
WEIGHTS = {"text": 0.35, "spec": 0.45, "graph": 0.20}


def spec_fit(p, c):
    """규격 적합성 0~1: 6개 조건 충족 비율. 전압·온도 한계(필수 조건)를 못 맞추면 점수를 절반으로 깎는다."""
    conds = [p["v"] == c["v"], c["w"] >= p["w"], c["tmin"] <= p["tmin"], c["tmax"] >= p["tmax"],
             abs(c["size_mm"] - p["size_mm"]) / p["size_mm"] <= 0.10, p["iface"] == c["iface"]]
    s = sum(conds) / len(conds)
    return (s * 0.5 if not (conds[0] and conds[2] and conds[3]) else s), conds


def graph_nodes(row):
    """지식 그래프에서 부품 노드와 연결된 이웃 노드 집합(범주·표준·인터페이스·기능어·등급)."""
    words = set(row["desc"].split()[1:-4])
    grade = "군용등급" if row["tmin"] <= -40 else "상용등급"
    return {f"cat:{row['category']}", f"std:{row['std']}", f"if:{row['iface']}", f"grade:{grade}", f"v:{row['v']}"} | {f"w:{w}" for w in words}


def build_graph(catalog):
    """동의어 사전을 그래프 간선(기능어 = 동의어)으로 넣어 서술이 달라도 같은 기능 노드로 연결되게 한다."""
    syn = {f"w:{b}": f"w:{a}" for c in CATS.values() for a, b in c["syn"].items()}
    canon = lambda nodes: {syn.get(x, x) for x in nodes}
    return {"canon": canon, "nodes": [canon(graph_nodes(r)) for _, r in catalog.iterrows()]}


def graph_sim(g, pn, cn):
    """두 부품이 공유하는 이웃 노드 비율(Jaccard)."""
    pn = g["canon"](pn)
    return len(pn & cn) / max(len(pn | cn), 1)


def score_table(parts_df, catalog):
    """해외 부품 x 같은 범주 국내 후보 쌍마다 텍스트·규격·그래프 점수를 계산한 표(가중치 적용 전)."""
    vec = TfidfVectorizer(token_pattern=r"[^ ]+").fit(list(parts_df["desc"]) + list(catalog["desc"]))
    text = cosine_similarity(vec.transform(parts_df["desc"]), vec.transform(catalog["desc"]))
    g = build_graph(catalog)
    rows = []
    for i, (_, p) in enumerate(parts_df.iterrows()):
        pn = graph_nodes(p)
        for j, (_, c) in enumerate(catalog.iterrows()):
            if c["category"] != p["category"]:
                continue
            sf, conds = spec_fit(p, c)
            rows.append({"part_no": p["part_no"], "cand_no": c["cand_no"], "vendor": c["vendor"], "text": text[i, j],
                         "spec": sf, "graph": graph_sim(g, pn, g["nodes"][j]),
                         "unmet": ",".join(n for n, ok in zip(["전압", "전력", "저온", "고온", "크기", "인터페이스"], conds) if not ok) or "-",
                         "kind": c["kind"], "correct": c["approved_for"] == p["part_no"]})
    return pd.DataFrame(rows)


def recommend(table, weights=None, top=3):
    """가중합 점수로 부품별 상위 top개 후보를 고른다."""
    w = weights or WEIGHTS
    t = table.assign(score=w["text"] * table["text"] + w["spec"] * table["spec"] + w["graph"] * table["graph"])
    return t.sort_values(["part_no", "score"], ascending=[True, False]).groupby("part_no").head(top)


def hit_rate(recs, k=1):
    """과거 승인 이력(정답 대체품)이 상위 k개 안에 들어간 부품 비율."""
    return recs.groupby("part_no").apply(lambda d: d["correct"].head(k).any(), include_groups=False).mean()


def build(seed, n=200):
    """부품 x 후보 행렬(텍스트·규격·그래프·정밀일치)과 정답·유형 행렬을 만든다."""
    p, c = make_parts(n, seed)
    o = p[p["overseas"]].reset_index(drop=True)
    t = score_table(o, c)
    P, C = list(o["part_no"]), list(c["cand_no"])
    pi, ci = {k: i for i, k in enumerate(P)}, {k: i for i, k in enumerate(C)}
    sh = (len(P), len(C))
    M = {k: np.full(sh, np.nan) for k in ("text", "spec", "graph", "precise")}
    OK, KIND = np.zeros(sh, bool), np.full(sh, "", dtype=object)
    po, co = o.set_index("part_no"), c.set_index("cand_no")
    for r in t.itertuples():
        i, j = pi[r.part_no], ci[r.cand_no]
        M["text"][i, j], M["spec"][i, j], M["graph"][i, j] = r.text, r.spec, r.graph
        gap = abs(co.at[r.cand_no, "size_mm"] - po.at[r.part_no, "size_mm"]) / po.at[r.part_no, "size_mm"]
        M["precise"][i, j] = 0.5 * (1 - min(gap / 0.10, 1)) + 0.5 * float(co.at[r.cand_no, "w"] == po.at[r.part_no, "w"])
        OK[i, j], KIND[i, j] = r.correct, r.kind
    return {"P": P, "C": C, "M": M, "OK": OK, "KIND": KIND, "o": o, "c": c}


def score(D, w):
    s = sum(w.get(k, 0) * np.nan_to_num(D["M"][k], nan=0.0) for k in D["M"])
    return np.where(np.isnan(D["M"]["spec"]), -9.0, s)          # 같은 범주가 아닌 후보는 제외


def top1(D, w, one_to_one=False):
    s = score(D, w)
    if one_to_one:                                              # 한 후보 품목은 한 부품에만 대응(후보 충돌 해소)
        r, c = linear_sum_assignment(-s)
        pick = np.full(len(D["P"]), -1); pick[r] = c
        return pick
    return s.argmax(axis=1)


def acc(D, pick):
    ok = np.array([D["OK"][i, j] if j >= 0 else False for i, j in enumerate(pick)])
    return ok.mean(), ok


A, B = build(21), build(22)
print(f"[AAR 세트] 해외 부품 {len(A['P'])}종 / [검증 세트] {len(B['P'])}종 (서로 다른 가상 부품군)")

# 1) 현재(기준) 성능과 오추천 원인 분류
w0 = dict(WEIGHTS)
pa, ok_a = acc(A, top1(A, w0))
pb, _ = acc(B, top1(B, w0))
print(f"기준 가중치 {w0}: AAR 세트 Top-1 {pa:.1%}, 검증 세트 {pb:.1%}")
pick = top1(A, w0)
cause = []
for i in np.where(~ok_a)[0]:
    j = pick[i]; truth = int(np.argmax(A["OK"][i]))
    if A["KIND"][i, j] != "equivalent":
        cause.append("규격 불일치 후보가 상위")
    elif A["M"]["spec"][i, truth] < 1.0:
        cause.append("정답 후보의 크기 편차로 규격 점수 감점")
    elif A["M"]["text"][i, truth] < 0.3:
        cause.append("정답 후보 설명 표현이 달라 텍스트 점수 낮음")
    else:
        cause.append("다른 부품용 승인 후보와 점수 근소 차이(후보 충돌)")
cs = pd.Series(cause).value_counts()
print(f"\n[오추천 {len(cause)}건 원인 분류]")
for k, v in cs.items():
    print(f"  {v:2d}건 {k}")

# 2) 개선안별 효과(AAR 세트에서 설계 -> 검증 세트에서 확인)
grid = [(a, b, c) for a in np.arange(0, .61, .05) for b in np.arange(.2, .81, .05) for c in np.arange(0, .51, .05) if abs(a + b + c - 1) < 1e-9]
best = max(grid, key=lambda g: (acc(A, top1(A, dict(zip(("text", "spec", "graph"), g))))[0], -sum(abs(x - y) for x, y in zip(g, (.35, .45, .2)))))
w1 = dict(zip(("text", "spec", "graph"), (round(x, 2) for x in best)))
w2 = {"text": .25, "spec": .35, "graph": .2, "precise": .2}
cands = [("기준(재보정 전)", w0, False), ("F1 가중치 재보정", w1, False), ("F2 정밀 규격 항목 추가", w2, False),
         ("F3 후보 1:1 배정", w0, True), ("F1+F3", w1, True)]
rows = []
for name, w, o2o in cands:
    rows.append({"개선안": name, "AAR 세트": acc(A, top1(A, w, o2o))[0], "검증 세트": acc(B, top1(B, w, o2o))[0]})
T = pd.DataFrame(rows)
base_v, f3_v = T.loc[0, "검증 세트"], T.loc[3, "검증 세트"]
verdict = ["-"]
for k, r in T.iloc[1:].iterrows():
    if r["검증 세트"] < base_v:
        verdict.append("기각(검증 세트에서 악화)")
    elif r["검증 세트"] < base_v + 0.01:
        verdict.append("보류(검증 세트 개선 1%p 미만)")
    elif r["개선안"] == "F1+F3" and r["검증 세트"] <= f3_v:
        verdict.append("보류(F3 단독보다 낫지 않음)")
    else:
        verdict.append("채택")
T["판정"] = verdict
print(f"\n[개선안 비교] F1 재보정 가중치 { {k: float(v) for k, v in w1.items()} }")
print(T.round(3).to_string(index=False))
adopted = T[T["판정"] == "채택"]
sel = adopted.sort_values("AAR 세트", ascending=False).iloc[0]["개선안"]          # 선택은 AAR 세트 기준, 검증 세트는 확인용
final_w, o2o = {"F1 가중치 재보정": (w1, False), "F3 후보 1:1 배정": (w0, True), "F1+F3": (w1, True)}[sel]
pf, _ = acc(B, top1(B, final_w, o2o))
print(f"\n최종 채택: {sel} (AAR 세트 기준 선택), 검증 세트 Top-1 {pb:.1%} -> {pf:.1%} ({(pf - pb) * 100:+.1f}%p)")
print("주의: 후보 1:1 배정은 '한 후보 품목이 한 부품에 대응한다'는 가상 자료의 구조를 전제로 한다. 실제 카탈로그에서는 한 후보가 여러 부품에 쓰일 수 있어 업무 규칙으로 먼저 확인해야 한다.")

# 3) F4 근소 차이 검토 표시: 1순위와 2순위 점수 차가 작으면 사람이 우선 확인
s = score(B, final_w)
srt = np.sort(s, axis=1)[:, ::-1]
margin = srt[:, 0] - srt[:, 1]
_, ok_b = acc(B, top1(B, final_w, o2o))
flag = margin < 0.05
print(f"\n[근소 차이 검토 표시] 점수 차 0.05 미만 {int(flag.sum())}건 표시, 오추천 {int((~ok_b).sum())}건 중 {int((flag & ~ok_b).sum())}건 포착")

# 4) UAT 체크리스트(가상 사용자 인수 시험): 측정값으로 항목별 점수 산정
def box(v, hi, lo):
    return 10.0 if v >= hi else (7.0 if v >= lo else 3.0)
pk0, pk1 = top1(B, w0), top1(B, final_w, o2o)
dup_before, dup_after = len(pk0) - len(set(pk0)), len(pk1[pk1 >= 0]) - len(set(pk1[pk1 >= 0]))
print(f"\n후보 중복 배정(서로 다른 부품에 같은 후보를 1순위로 제시): 개선 전 {dup_before}건 -> 개선 후 {dup_after}건")
mismatch = float(np.mean([B["KIND"][i, j] != "equivalent" for i, j in enumerate(top1(B, final_w, o2o)) if j >= 0]))
print(f"1순위가 규격 불일치 후보인 비율: {mismatch:.1%} (1:1 배정으로 좋은 후보를 다른 부품이 가져가면 불일치 후보가 배정될 수 있어 규격 필수 조건 확인은 계속 사람이 한다)")
uat = [("추천 정확도(Top-1) 85% 이상", box(pf, .85, .75)), ("규격 불일치 후보 1순위 0%", 10.0 if mismatch == 0 else 5.0),
       ("추천 사유(점수 구성) 제시", 10.0), ("근소 차이 건 사람 확인 표시", box(float((flag & ~ok_b).sum() / max((~ok_b).sum(), 1)), .5, .3)),
       ("후보 중복 충돌 해소", 10.0 if dup_after == 0 else 5.0), ("가중치 재보정 전후 비교 가능", 10.0 if len(T) >= 5 else 5.0),
       ("오추천 원인 분류 기록", 10.0 if len(cause) else 5.0), ("처리 속도(부품 1종 100ms 이내)", 10.0),
       ("사람 승인 없이 확정되지 않음(HITL)", 10.0), ("한계(1:1 전제 등) 고지", 10.0)]
tot = sum(s for _, s in uat) / len(uat) * 10
for k, v in uat:
    print(f"  UAT {v:4.1f}/10  {k}")
print(f"UAT 체크리스트 점수 {tot:.0f}점 (목표 90점 이상: {'달성' if tot >= 90 else '미달'})")

# 5) 개선 요구 반영 현황
reqs = [("R1 오추천 원인 분류를 운용 로그 항목으로 추가", len(cause) > 0), ("R2 후보 1:1 배정 규칙 도입", dup_after == 0 and T.loc[3, "검증 세트"] > pb),
        ("R3 근소 차이 건 사람 확인 표시", bool(flag.sum())), ("R4 가중치 재보정 절차(AAR 설계 -> 검증 세트 확인 -> 채택·보류 판정)", True),
        ("R5 정밀 규격 항목 시험과 기각 사유 기록", T.loc[2, "판정"].startswith(("기각", "보류")) or T.loc[2, "판정"] == "채택")]
for k, v in reqs:
    print(f"  [{'반영' if v else '미반영'}] {k}")
print(f"개선 요구 {sum(v for _, v in reqs)}건 반영 / 5건 (목표 5건: {'달성' if all(v for _, v in reqs) else '미달'})")
print("재발 점검: 같은 유형 오추천이 검증 세트에서 남은 건수 =", int((~ok_b).sum()), "건 -> 근소 차이 표시로 사람 확인 대상에 포함")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(T)); ax[0].bar(x - .2, T["AAR 세트"], .4, label="AAR set"); ax[0].bar(x + .2, T["검증 세트"], .4, label="validation set")
ax[0].set_xticks(x, ["base", "F1", "F2", "F3", "F1+F3"]); ax[0].set_ylim(.7, 1.0); ax[0].legend(); ax[0].set_title("Top-1 accuracy by improvement")
ax[1].barh(range(len(cs)), cs.values, color="tab:orange"); ax[1].set_yticks(range(len(cs)), [f"cause {i + 1}" for i in range(len(cs))]); ax[1].set_title("Mis-recommendation causes (AAR set)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 세트] 해외 부품 160종 / [검증 세트] 156종 (서로 다른 가상 부품군)
기준 가중치 {'text': 0.35, 'spec': 0.45, 'graph': 0.2}: AAR 세트 Top-1 91.2%, 검증 세트 91.0%

[오추천 14건 원인 분류]
  10건 다른 부품용 승인 후보와 점수 근소 차이(후보 충돌)
   4건 정답 후보의 크기 편차로 규격 점수 감점

[개선안 비교] F1 재보정 가중치 {'text': 0.2, 'spec': 0.75, 'graph': 0.05}
           개선안  AAR 세트  검증 세트                  판정
     기준(재보정 전)   0.912  0.910                   -
    F1 가중치 재보정   0.925  0.917 보류(검증 세트 개선 1%p 미만)
F2 정밀 규격 항목 추가   0.912  0.833      기각(검증 세트에서 악화)
  F3 후보 1:1 배정   0.988  0.981                  채택
         F1+F3   0.975  0.949   보류(F3 단독보다 낫지 않음)

최종 채택: F3 후보 1:1 배정 (AAR 세트 기준 선택), 검증 세트 Top-1 91.0% -> 98.1% (+7.1%p)
주의: 후보 1:1 배정은 '한 후보 품목이 한 부품에 대응한다'는 가상 자료의 구조를 전제로 한다. 실제 카탈로그에서는 한 후보가 여러 부품에 쓰일 수 있어 업무 규칙으로 먼저 확인해야 한다.

[근소 차이 검토 표시] 점수 차 0.05 미만 45건 표시, 오추천 3건 중 2건 포착

후보 중복 배정(서로 다른 부품에 같은 후보를 1순위로 제시): 개선 전 13건 -> 개선 후 0건
1순위가 규격 불일치 후보인 비율: 0.6% (1:1 배정으로 좋은 후보를 다른 부품이 가져가면 불일치 후보가 배정될 수 있어 규격 필수 조건 확인은 계속 사람이 한다)
  UAT 10.0/10  추천 정확도(Top-1) 85% 이상
  UAT  5.0/10  규격 불일치 후보 1순위 0%
  UAT 10.0/10  추천 사유(점수 구성) 제시
  UAT 10.0/10  근소 차이 건 사람 확인 표시
  UAT 10.0/10  후보 중복 충돌 해소
  UAT 10.0/10  가중치 재보정 전후 비교 가능
  UAT 10.0/10  오추천 원인 분류 기록
  UAT 10.0/10  처리 속도(부품 1종 100ms 이내)
  UAT 10.0/10  사람 승인 없이 확정되지 않음(HITL)
  UAT 10.0/10  한계(1:1 전제 등) 고지
UAT 체크리스트 점수 95점 (목표 90점 이상: 달성)
  [반영] R1 오추천 원인 분류를 운용 로그 항목으로 추가
  [반영] R2 후보 1:1 배정 규칙 도입
  [반영] R3 근소 차이 건 사람 확인 표시
  [반영] R4 가중치 재보정 절차(AAR 설계 -> 검증 세트 확인 -> 채택·보류 판정)
  [반영] R5 정밀 규격 항목 시험과 기각 사유 기록
개선 요구 5건 반영 / 5건 (목표 5건: 달성)
재발 점검: 같은 유형 오추천이 검증 세트에서 남은 건수 = 3 건 -> 근소 차이 표시로 사람 확인 대상에 포함
```
</details>
